# Notebook 3: Tree Hyperparameters — Controlling Complexity

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2.5 hours  
**Theme:** House Price Prediction (regression + binary classification: above/below median)  
**Series:** Week 9 — Tree-Based Methods & Gradient Boosting

---

## Why This Matters

A decision tree with no constraints will memorize every training sample — achieving perfect training accuracy while failing completely on new data. Hyperparameters are the levers that let you control the bias–variance trade-off. Understanding *what each lever does* and *how they interact* is essential before you can reliably tune any tree-based model (including Random Forests and Gradient Boosting).

---

## Real-World Analogy First: The Garden Hedge

Imagine growing a garden hedge from scratch:

- **Unconstrained growth:** Let the hedge grow wild for a whole season. It perfectly follows every bump and dip in the terrain — every stone, every weed, every accidental footprint. It looks like chaos. It has memorised the terrain (overfitting).
- **Pruning rules:** Now you apply constraints:
  - *"No branch shorter than 10 cm"* → `min_samples_leaf`: no leaf node allowed to hold fewer than 10 training samples
  - *"No branch thicker than your thumb"* → `min_samples_split`: don't split a node unless it has enough samples to justify the split
  - *"Maximum 3 layers tall"* → `max_depth`: the hedge can only have 3 levels of branching

The rules shape the hedge into something **generalizable** — it captures the major contours of the terrain without memorizing every pebble.

Decision tree hyperparameters are exactly these rules. They don't change the tree's learning algorithm — they constrain *how far* the algorithm is allowed to go.

## Hyperparameter Reference

| Hyperparameter | Plain English | Effect on Tree | Typical Range |
|---|---|---|---|
| `max_depth` | How many layers of questions | Limits tree height | 3–10 |
| `min_samples_split` | Min samples to bother splitting a node | Prunes small internal nodes | 2–200 |
| `min_samples_leaf` | Min samples every leaf must hold | Prevents single-outlier leaves | 1–50 |
| `max_features` | Features considered at each split | Controls randomness (for ensembles) | sqrt, log2, None |
| `max_leaf_nodes` | Hard cap on total leaf nodes | Alternative to max_depth | 10–200 |

### The Formulas

**Impurity (MSE for regression):**
$$\text{MSE}_{node} = \frac{1}{N_{node}} \sum_{i \in node} (y_i - \bar{y}_{node})^2$$

**Impurity reduction at a split:**
$$\Delta I = I(parent) - \frac{N_{left}}{N_{parent}} I(left) - \frac{N_{right}}{N_{parent}} I(right)$$

A split only happens if $\Delta I > 0$ *and* the constraints are satisfied:
- $N_{parent} \geq$ `min_samples_split`
- $N_{left} \geq$ `min_samples_leaf` **and** $N_{right} \geq$ `min_samples_leaf`
- current depth $<$ `max_depth`

### Interaction Insight

> **These hyperparameters interact.** A tree with `max_depth=10` AND `min_samples_leaf=50` on 500 samples is very different from either constraint alone. `max_depth=10` allows up to $2^{10} = 1024$ leaves — but `min_samples_leaf=50` means each leaf needs 50 samples, so with 500 total samples you can have at most $500/50 = 10$ leaves. The binding constraint is `min_samples_leaf`.

## Section 1: Setup and Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully.")

In [ ]:
# --- Synthetic House Price Dataset: 800 samples, 8 features ---
np.random.seed(42)
N = 800

sqft         = np.random.normal(1800, 500, N).clip(500, 4000)
bedrooms     = np.random.randint(1, 7, N).astype(float)
bathrooms    = np.random.uniform(1, 4, N)
lot_size     = np.random.normal(7000, 2000, N).clip(1000, 20000)
age          = np.random.randint(0, 80, N).astype(float)
garage       = np.random.randint(0, 4, N).astype(float)
school_score = np.random.uniform(3, 10, N)
dist_center  = np.random.uniform(1, 30, N)

# Price model with noise
price = (
    150 * sqft / 1000
    + 5000 * bedrooms
    + 8000 * bathrooms
    + 2 * lot_size
    - 500 * age
    + 4000 * garage
    + 3000 * school_score
    - 1500 * dist_center
    + np.random.normal(0, 15000, N)
    + 80000
).clip(50000, 800000)

feature_names = ['sqft', 'bedrooms', 'bathrooms', 'lot_size', 'age', 'garage', 'school_score', 'dist_center']

df = pd.DataFrame({
    'sqft': sqft, 'bedrooms': bedrooms, 'bathrooms': bathrooms,
    'lot_size': lot_size, 'age': age, 'garage': garage,
    'school_score': school_score, 'dist_center': dist_center,
    'price': price
})

# Binary target: above/below median price
median_price = df['price'].median()
df['above_median'] = (df['price'] > median_price).astype(int)

print(f"Dataset: {df.shape[0]} samples, {len(feature_names)} features")
print(f"Price range: ${df['price'].min():,.0f} – ${df['price'].max():,.0f}")
print(f"Median price: ${median_price:,.0f}")
print(f"Above median: {df['above_median'].mean():.1%}")
df.describe().round(1)

In [ ]:
# Train / Validation / Test Split
X = df[feature_names].values
y_reg = df['price'].values
y_clf = df['above_median'].values

# 60% train, 20% val, 20% test
X_temp, X_test, y_temp_r, y_test_r = train_test_split(X, y_reg,  test_size=0.2, random_state=42)
X_train, X_val, y_train_r, y_val_r  = train_test_split(X_temp, y_temp_r, test_size=0.25, random_state=42)

_, _,         y_temp_c, y_test_c    = train_test_split(X, y_clf,  test_size=0.2, random_state=42)
_, _,         y_train_c, y_val_c    = train_test_split(X_temp, y_temp_c, test_size=0.25, random_state=42)
# (same X splits, different y)

print(f"Train:      {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test:       {X_test.shape[0]} samples")

## Section 2: `max_depth` — Controlling Tree Height

**Plain English:** `max_depth` is the most interpretable lever. Each additional depth level allows the tree to ask one more follow-up question. At depth=1, you get a single split (a "decision stump"). At unlimited depth, the tree will keep splitting until every leaf is pure — memorizing the training data.

**Key insight:** The optimal depth is where validation error bottoms out. After that point, more depth only buys training performance at the cost of generalization.

In [ ]:
# Sweep max_depth from 1 to 20
depths = list(range(1, 21))
train_mse_depth, val_mse_depth = [], []
train_r2_depth,  val_r2_depth  = [], []
n_leaves_depth = []

for d in depths:
    tree = DecisionTreeRegressor(max_depth=d, random_state=42)
    tree.fit(X_train, y_train_r)
    
    tr_pred = tree.predict(X_train)
    vl_pred = tree.predict(X_val)
    
    train_mse_depth.append(mean_squared_error(y_train_r, tr_pred))
    val_mse_depth.append(mean_squared_error(y_val_r,   vl_pred))
    train_r2_depth.append(r2_score(y_train_r, tr_pred))
    val_r2_depth.append(r2_score(y_val_r,   vl_pred))
    n_leaves_depth.append(tree.get_n_leaves())

train_rmse_depth = np.sqrt(train_mse_depth)
val_rmse_depth   = np.sqrt(val_mse_depth)
best_depth_idx   = np.argmin(val_rmse_depth)
best_depth       = depths[best_depth_idx]

print(f"Optimal max_depth (by val RMSE): {best_depth}")
print(f"  Train RMSE at optimal depth:   ${train_rmse_depth[best_depth_idx]:,.0f}")
print(f"  Val   RMSE at optimal depth:   ${val_rmse_depth[best_depth_idx]:,.0f}")
print(f"  Val   R² at optimal depth:     {val_r2_depth[best_depth_idx]:.4f}")
print(f"  Leaves at optimal depth:       {n_leaves_depth[best_depth_idx]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: RMSE vs depth
ax = axes[0]
ax.plot(depths, train_rmse_depth, 'o-', color='steelblue', label='Train RMSE', linewidth=2)
ax.plot(depths, val_rmse_depth,   's-', color='tomato',    label='Val RMSE',   linewidth=2)
ax.axvline(best_depth, color='darkgreen', linestyle='--', linewidth=1.8, label=f'Optimal depth={best_depth}')
ax.fill_between(depths, train_rmse_depth, val_rmse_depth,
                where=[v > t for v, t in zip(val_rmse_depth, train_rmse_depth)],
                alpha=0.12, color='tomato', label='Overfitting gap')
ax.set_xlabel('max_depth')
ax.set_ylabel('RMSE ($)')
ax.set_title('RMSE vs max_depth')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

# Right: R² vs depth
ax = axes[1]
ax.plot(depths, train_r2_depth, 'o-', color='steelblue', label='Train R²', linewidth=2)
ax.plot(depths, val_r2_depth,   's-', color='tomato',    label='Val R²',   linewidth=2)
ax.axvline(best_depth, color='darkgreen', linestyle='--', linewidth=1.8, label=f'Optimal depth={best_depth}')
ax.set_xlabel('max_depth')
ax.set_ylabel('R²')
ax.set_title('R² vs max_depth')
ax.legend()

plt.suptitle('Effect of max_depth on House Price Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nInterpretation:")
print(f"  depth=1 (stump): Train R²={train_r2_depth[0]:.3f}, Val R²={val_r2_depth[0]:.3f}  ← underfitting")
print(f"  depth={best_depth} (optimal): Train R²={train_r2_depth[best_depth_idx]:.3f}, Val R²={val_r2_depth[best_depth_idx]:.3f}  ← best generalization")
print(f"  depth=20 (deep): Train R²={train_r2_depth[-1]:.3f}, Val R²={val_r2_depth[-1]:.3f}  ← overfitting")

## Section 3: `min_samples_split` — Minimum Samples to Split a Node

**Plain English:** Before splitting a node, the algorithm checks: "Does this node have at least `min_samples_split` samples?" If not, it becomes a leaf regardless of how impure it is.

- Default is 2 — extremely permissive, allows splitting with just 2 samples
- Higher values force the tree to only split where there's sufficient data evidence
- Has a **smoothing effect** on very noisy parts of the feature space

In [ ]:
# Sweep min_samples_split: 2 to 200
mss_values = list(range(2, 201, 4))  # step 4 for speed
train_rmse_mss, val_rmse_mss = [], []
train_r2_mss,   val_r2_mss   = [], []
n_leaves_mss = []

for mss in mss_values:
    tree = DecisionTreeRegressor(min_samples_split=mss, random_state=42)
    tree.fit(X_train, y_train_r)
    tr_pred = tree.predict(X_train)
    vl_pred = tree.predict(X_val)
    train_rmse_mss.append(np.sqrt(mean_squared_error(y_train_r, tr_pred)))
    val_rmse_mss.append(np.sqrt(mean_squared_error(y_val_r,   vl_pred)))
    train_r2_mss.append(r2_score(y_train_r, tr_pred))
    val_r2_mss.append(r2_score(y_val_r,   vl_pred))
    n_leaves_mss.append(tree.get_n_leaves())

best_mss_idx = np.argmin(val_rmse_mss)
best_mss = mss_values[best_mss_idx]
print(f"Optimal min_samples_split: {best_mss}")
print(f"  Val RMSE: ${val_rmse_mss[best_mss_idx]:,.0f}")
print(f"  Val R²:   {val_r2_mss[best_mss_idx]:.4f}")
print(f"  Leaves:   {n_leaves_mss[best_mss_idx]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.plot(mss_values, train_rmse_mss, 'o-', color='steelblue', markersize=4, label='Train RMSE', linewidth=2)
ax.plot(mss_values, val_rmse_mss,   's-', color='tomato',    markersize=4, label='Val RMSE',   linewidth=2)
ax.axvline(best_mss, color='darkgreen', linestyle='--', linewidth=1.8, label=f'Optimal={best_mss}')
ax.set_xlabel('min_samples_split')
ax.set_ylabel('RMSE ($)')
ax.set_title('RMSE vs min_samples_split')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

ax2 = axes[1]
ax2.plot(mss_values, n_leaves_mss, 'o-', color='purple', markersize=4, linewidth=2)
ax2.axvline(best_mss, color='darkgreen', linestyle='--', linewidth=1.8, label=f'Optimal={best_mss}')
ax2.set_xlabel('min_samples_split')
ax2.set_ylabel('Number of Leaves')
ax2.set_title('Tree Size vs min_samples_split')
ax2.legend()

plt.suptitle('Effect of min_samples_split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nNote: Val RMSE stabilizes around min_samples_split={best_mss}.")
print(f"Beyond that, we're just making the tree simpler without improving generalization.")

## Section 4: `min_samples_leaf` — Minimum Samples per Leaf

**Plain English:** Every leaf node must contain **at least** this many training samples. This is subtly different from `min_samples_split`:

- `min_samples_split` checks the *parent* node before splitting
- `min_samples_leaf` checks *both child nodes* after the split would occur
- `min_samples_leaf` is therefore **more aggressive** — it prevents splits where one child would be tiny, even if the parent is large

Think of it like requiring a minimum **population density** in every region of the feature space.

In [ ]:
# Sweep min_samples_leaf: 1 to 100
msl_values = list(range(1, 101, 2))  # step 2
train_rmse_msl, val_rmse_msl = [], []
train_r2_msl,   val_r2_msl   = [], []
n_leaves_msl = []

for msl in msl_values:
    tree = DecisionTreeRegressor(min_samples_leaf=msl, random_state=42)
    tree.fit(X_train, y_train_r)
    tr_pred = tree.predict(X_train)
    vl_pred = tree.predict(X_val)
    train_rmse_msl.append(np.sqrt(mean_squared_error(y_train_r, tr_pred)))
    val_rmse_msl.append(np.sqrt(mean_squared_error(y_val_r,   vl_pred)))
    train_r2_msl.append(r2_score(y_train_r, tr_pred))
    val_r2_msl.append(r2_score(y_val_r,   vl_pred))
    n_leaves_msl.append(tree.get_n_leaves())

best_msl_idx = np.argmin(val_rmse_msl)
best_msl = msl_values[best_msl_idx]
print(f"Optimal min_samples_leaf: {best_msl}")
print(f"  Val RMSE: ${val_rmse_msl[best_msl_idx]:,.0f}")
print(f"  Val R²:   {val_r2_msl[best_msl_idx]:.4f}")
print(f"  Leaves:   {n_leaves_msl[best_msl_idx]}")

# --- Side by side comparison ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: RMSE comparison
ax = axes[0]
ax.plot(mss_values, val_rmse_mss, 'o-', color='steelblue', markersize=4,
        label='min_samples_split (val)', linewidth=2, alpha=0.8)
ax.plot(msl_values, val_rmse_msl, 's-', color='tomato', markersize=4,
        label='min_samples_leaf (val)', linewidth=2, alpha=0.8)
ax.set_xlabel('Hyperparameter value')
ax.set_ylabel('Validation RMSE ($)')
ax.set_title('min_samples_split vs min_samples_leaf\n(Validation RMSE)')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

# Right: Leaf count
ax = axes[1]
ax.plot(mss_values, n_leaves_mss, 'o-', color='steelblue', markersize=4,
        label='min_samples_split', linewidth=2, alpha=0.8)
ax.plot(msl_values, n_leaves_msl, 's-', color='tomato', markersize=4,
        label='min_samples_leaf', linewidth=2, alpha=0.8)
ax.set_xlabel('Hyperparameter value')
ax.set_ylabel('Number of Leaves')
ax.set_title('Tree Size: min_samples_split vs min_samples_leaf')
ax.legend()

plt.suptitle('Comparing min_samples_split vs min_samples_leaf', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("\nKey observation: min_samples_leaf reduces tree size faster than min_samples_split")
print("for the same parameter value, because it constraints BOTH sides of every split.")

## Section 5: 2D Interaction — `max_depth` × `min_samples_leaf`

Since these hyperparameters **interact**, we need to look at them jointly. A 2D heatmap of validation R² reveals the landscape of good configurations.

In [ ]:
# 2D grid: max_depth x min_samples_leaf
depth_grid = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]
msl_grid   = [1, 2, 5, 10, 20, 30, 50, 75, 100]

val_r2_grid    = np.zeros((len(depth_grid), len(msl_grid)))
n_leaves_grid  = np.zeros((len(depth_grid), len(msl_grid)))

for i, d in enumerate(depth_grid):
    for j, msl in enumerate(msl_grid):
        tree = DecisionTreeRegressor(max_depth=d, min_samples_leaf=msl, random_state=42)
        tree.fit(X_train, y_train_r)
        val_r2_grid[i, j]   = r2_score(y_val_r, tree.predict(X_val))
        n_leaves_grid[i, j] = tree.get_n_leaves()

best_flat = np.argmax(val_r2_grid)
best_i, best_j = np.unravel_index(best_flat, val_r2_grid.shape)
print(f"Best val R²: {val_r2_grid[best_i, best_j]:.4f}")
print(f"  max_depth={depth_grid[best_i]}, min_samples_leaf={msl_grid[best_j]}")
print(f"  Leaves: {n_leaves_grid[best_i, best_j]:.0f}")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap 1: Validation R²
ax = axes[0]
im = ax.imshow(val_r2_grid, cmap='RdYlGn', aspect='auto',
               vmin=val_r2_grid.min(), vmax=val_r2_grid.max())
plt.colorbar(im, ax=ax, label='Validation R²')
ax.set_xticks(range(len(msl_grid)))
ax.set_xticklabels(msl_grid)
ax.set_yticks(range(len(depth_grid)))
ax.set_yticklabels(depth_grid)
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('max_depth')
ax.set_title('Validation R²: max_depth × min_samples_leaf')
# Annotate cells
for i in range(len(depth_grid)):
    for j in range(len(msl_grid)):
        color = 'white' if val_r2_grid[i, j] < (val_r2_grid.min() + val_r2_grid.max()) / 2 else 'black'
        ax.text(j, i, f'{val_r2_grid[i, j]:.2f}', ha='center', va='center',
                fontsize=7.5, color=color)
# Mark best
rect = plt.Rectangle((best_j - 0.5, best_i - 0.5), 1, 1, fill=False,
                      edgecolor='navy', linewidth=3)
ax.add_patch(rect)
ax.text(best_j, best_i - 0.38, 'BEST', ha='center', va='center',
        fontsize=8, color='navy', fontweight='bold')

# Heatmap 2: Number of leaves
ax = axes[1]
im2 = ax.imshow(n_leaves_grid, cmap='Blues', aspect='auto')
plt.colorbar(im2, ax=ax, label='Number of Leaves')
ax.set_xticks(range(len(msl_grid)))
ax.set_xticklabels(msl_grid)
ax.set_yticks(range(len(depth_grid)))
ax.set_yticklabels(depth_grid)
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('max_depth')
ax.set_title('Tree Size (Leaves): max_depth × min_samples_leaf')
for i in range(len(depth_grid)):
    for j in range(len(msl_grid)):
        ax.text(j, i, f'{n_leaves_grid[i, j]:.0f}', ha='center', va='center',
                fontsize=7.5, color='black')

plt.suptitle('2D Hyperparameter Interaction: max_depth × min_samples_leaf', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 6: Tree Size vs Performance

Number of leaves is a single number that integrates the effect of all size-controlling hyperparameters. Plotting validation R² vs leaf count shows whether complexity is actually helping.

In [ ]:
# Collect all (n_leaves, val_r2) pairs from the 2D grid
all_leaves  = n_leaves_grid.flatten()
all_val_r2  = val_r2_grid.flatten()

# Also: vary ONLY max_depth (other defaults)
leaves_by_depth = n_leaves_depth
r2_by_depth     = val_r2_depth

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter: all grid combinations
ax = axes[0]
scatter = ax.scatter(all_leaves, all_val_r2, c=all_val_r2, cmap='RdYlGn',
                     s=60, alpha=0.8, edgecolors='gray', linewidths=0.5)
plt.colorbar(scatter, ax=ax, label='Val R²')
ax.set_xlabel('Number of Leaves')
ax.set_ylabel('Validation R²')
ax.set_title('Val R² vs Tree Size\n(all max_depth × min_samples_leaf combos)')

# Line: varying only max_depth
ax = axes[1]
ax.plot(leaves_by_depth, r2_by_depth, 'o-', color='steelblue', linewidth=2, markersize=6)
for i, (l, r, d) in enumerate(zip(leaves_by_depth, r2_by_depth, depths)):
    if d in [1, 3, best_depth, 10, 20]:
        ax.annotate(f'd={d}', (l, r), textcoords='offset points',
                    xytext=(5, 5), fontsize=8, color='darkblue')
ax.set_xlabel('Number of Leaves')
ax.set_ylabel('Validation R²')
ax.set_title('Val R² vs Tree Size\n(varying only max_depth)')

plt.suptitle('Does More Complexity Help?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Observation: R² initially improves with more leaves, then plateaus or declines.")
print("More leaves ≠ more accuracy beyond the sweet spot.")

## Section 7: Classification — Accuracy vs Hyperparameters

The same hyperparameter effects apply to the binary classification task (above/below median price). Let's verify the patterns hold.

In [ ]:
# Classification sweep: max_depth
train_acc_depth, val_acc_depth = [], []
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train_c)
    train_acc_depth.append(accuracy_score(y_train_c, clf.predict(X_train)))
    val_acc_depth.append(accuracy_score(y_val_c,   clf.predict(X_val)))

# Classification sweep: min_samples_leaf
train_acc_msl, val_acc_msl = [], []
for msl in msl_values:
    clf = DecisionTreeClassifier(min_samples_leaf=msl, random_state=42)
    clf.fit(X_train, y_train_c)
    train_acc_msl.append(accuracy_score(y_train_c, clf.predict(X_train)))
    val_acc_msl.append(accuracy_score(y_val_c,   clf.predict(X_val)))

best_clf_depth = depths[np.argmax(val_acc_depth)]
best_clf_msl   = msl_values[np.argmax(val_acc_msl)]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.plot(depths, train_acc_depth, 'o-', color='steelblue', label='Train Accuracy', linewidth=2)
ax.plot(depths, val_acc_depth,   's-', color='tomato',    label='Val Accuracy',   linewidth=2)
ax.axvline(best_clf_depth, color='darkgreen', linestyle='--', linewidth=1.8,
           label=f'Optimal depth={best_clf_depth}')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Classification Accuracy vs max_depth')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

ax = axes[1]
ax.plot(msl_values, train_acc_msl, 'o-', color='steelblue', markersize=4, label='Train Accuracy', linewidth=2)
ax.plot(msl_values, val_acc_msl,   's-', color='tomato',    markersize=4, label='Val Accuracy',   linewidth=2)
ax.axvline(best_clf_msl, color='darkgreen', linestyle='--', linewidth=1.8,
           label=f'Optimal msl={best_clf_msl}')
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('Accuracy')
ax.set_title('Classification Accuracy vs min_samples_leaf')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.suptitle('Classification: Above/Below Median Price', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Best classification depth: {best_clf_depth} (val acc = {max(val_acc_depth):.1%})")
print(f"Best classification msl:   {best_clf_msl}  (val acc = {max(val_acc_msl):.1%})")

## Section 8: GridSearchCV — Systematic Hyperparameter Tuning

In [ ]:
# GridSearchCV on regression task
param_grid = {
    'max_depth':        [3, 4, 5, 6, 7, 8, 10],
    'min_samples_leaf': [1, 5, 10, 20, 30, 50],
    'min_samples_split': [2, 10, 20]
}

# Use train+val combined for CV
X_cv = np.vstack([X_train, X_val])
y_cv = np.concatenate([y_train_r, y_val_r])

grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    return_train_score=True
)
grid_search.fit(X_cv, y_cv)

print("=" * 55)
print("GridSearchCV Results (5-fold CV on train+val)")
print("=" * 55)
print(f"Best parameters:   {grid_search.best_params_}")
print(f"Best CV R²:        {grid_search.best_score_:.4f}")

# Compare: best grid vs visual inspection
best_grid_tree = grid_search.best_estimator_
best_grid_tree.fit(X_train, y_train_r)

visual_tree = DecisionTreeRegressor(
    max_depth=best_depth,
    random_state=42
)
visual_tree.fit(X_train, y_train_r)

print("\n" + "=" * 55)
print("Comparison: GridSearch vs Visual Inspection")
print("=" * 55)
print(f"{'Method':<25} {'Val RMSE':>12} {'Val R²':>10} {'Leaves':>8}")
print("-" * 55)

for name, tree in [("Visual (depth sweep)", visual_tree), ("GridSearchCV", best_grid_tree)]:
    pred = tree.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val_r, pred))
    r2   = r2_score(y_val_r, pred)
    print(f"{name:<25} ${rmse:>10,.0f} {r2:>10.4f} {tree.get_n_leaves():>8}")

# Also test set performance
print("\n" + "=" * 55)
print("Test Set Performance (GridSearch best model)")
print("=" * 55)
best_grid_tree.fit(X_cv, y_cv)  # refit on full train+val
test_pred = best_grid_tree.predict(X_test)
print(f"Test RMSE: ${np.sqrt(mean_squared_error(y_test_r, test_pred)):,.0f}")
print(f"Test R²:   {r2_score(y_test_r, test_pred):.4f}")

In [ ]:
# Visualize best tree structure from GridSearch
best_vis_tree = DecisionTreeRegressor(**grid_search.best_params_, random_state=42)
best_vis_tree.fit(X_train, y_train_r)

actual_depth = best_vis_tree.get_depth()
cap_depth = min(actual_depth, 3)  # cap for readability

fig, ax = plt.subplots(figsize=(20, 7))
plot_tree(
    best_vis_tree,
    max_depth=cap_depth,
    feature_names=feature_names,
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax
)
ax.set_title(
    f"Best Tree from GridSearchCV (showing first {cap_depth} levels)\n"
    f"Full depth={actual_depth}, leaves={best_vis_tree.get_n_leaves()}\n"
    f"Params: {grid_search.best_params_}",
    fontsize=11
)
plt.tight_layout()
plt.show()

## Summary: Hyperparameter Cheat Sheet

| Hyperparameter | Too Small | Too Large | Typical Sweet Spot |
|---|---|---|---|
| `max_depth` | Underfits (misses patterns) | Overfits (memorises noise) | 3–8 |
| `min_samples_split` | Noisy splits allowed | Misses useful splits | 5–50 |
| `min_samples_leaf` | Single-outlier leaves | Under-responsive regions | 5–30 |
| `max_leaf_nodes` | Too coarse | Equivalent to no constraint | 20–200 |

**Rule of thumb for starting values:**
1. Start with `max_depth=5`, `min_samples_leaf=5`
2. Plot the validation curve while sweeping `max_depth` (1 to 20)
3. Fix the best `max_depth`, then sweep `min_samples_leaf`
4. Run GridSearchCV around the visually identified region
5. Always evaluate on a held-out test set (not the validation set used for tuning)

---
## Self-Check Questions

Answer these without running code first, then verify.

---

**Question 1:** You set `min_samples_leaf=50` on a dataset of 1000 samples. At most, how many leaves can the tree have?

<details>
<summary>Show Answer</summary>

**At most 20 leaves.**

Each leaf must contain at least 50 samples. With 1000 total training samples, the maximum number of leaves is $\lfloor 1000 / 50 \rfloor = 20$.

In practice, the tree may have fewer than 20 leaves because:
- The 1000 samples don't divide perfectly into equal groups of 50
- Some branches may terminate early due to purity (no impurity left to reduce)
- Other hyperparameters (e.g., `max_depth`) may impose tighter constraints first

This illustrates that `min_samples_leaf` places an **upper bound** on tree complexity that is directly tied to dataset size.

</details>

---

**Question 2:** A tree with `max_depth=5` has $2^5 = 32$ potential leaf nodes. Why might it actually have fewer leaves?

<details>
<summary>Show Answer</summary>

**Several reasons a tree with max_depth=5 may have fewer than 32 leaves:**

1. **Node purity:** If all samples in a node have the same label (classification) or zero variance (regression), there is nothing to split on — the node becomes a leaf even if depth < 5.

2. **min_samples_split / min_samples_leaf:** If a node has too few samples to split (below the threshold), it stops early.

3. **No useful split exists:** If all features have the same value in a node, no split can improve impurity.

4. **Sparse branches:** If one child of a split gets very few samples, subsequent splits in that sub-branch may be impossible.

`max_depth=5` sets a **ceiling**, not a guaranteed count. The actual tree shape depends on data distribution.

</details>

---

**Question 3:** With `min_samples_split=20` and `min_samples_leaf=10` — which is more restrictive? Is it possible for `min_samples_leaf` to be more restrictive than `min_samples_split`?

<details>
<summary>Show Answer</summary>

**It depends on the specific split, but `min_samples_leaf` is generally more restrictive in this example.**

`min_samples_split=20` says: *"Only split if the parent node has ≥ 20 samples."*

`min_samples_leaf=10` says: *"Only split if BOTH children will have ≥ 10 samples."*

Consider a node with 25 samples:
- `min_samples_split=20` → **allows** splitting (25 ≥ 20) ✓
- `min_samples_leaf=10` → only allows splitting if BOTH children get ≥ 10 samples. A 15/10 split is fine, but a 24/1 split is **rejected**.

So yes, `min_samples_leaf` can be more restrictive than `min_samples_split` when the split would be very uneven. In the hedge analogy: `min_samples_split` checks the branch *before* cutting; `min_samples_leaf` checks *both resulting branches* after the cut. This is why `min_samples_leaf` tends to produce smoother, more balanced trees.

**Key insight:** `min_samples_leaf=k` implicitly requires `min_samples_split ≥ 2k`. So `min_samples_leaf=10` is at least as restrictive as `min_samples_split=20` for the split condition.

</details>